# VL-JEPA (v1) — Vision-Language JEPA · Training

Trains the compact **VL-JEPA** model from the inference notebook. Training uses
**block masking** on image patches: the model predicts masked latent content
conditioned on the (visible patches + caption), pulling matching image/text
representations together in the shared space.

### What this notebook does
1. Load config and shrink the schedule for a quick demo (3 epochs).
2. Build the model, a dummy paired dataset, and a `BlockMaskCollator`.
3. Run `VLJEPATrainer` and save checkpoints to `vljepa_model/`.

After training, point the **inference** notebook's `model_id` at the saved
checkpoint (e.g. `vljepa_model/checkpoint_epoch0003.pt`) to see the similarity
matrix's diagonal strengthen.


In [2]:
# Confirm GPU availability before training.
!nvidia-smi

Sun Jul  5 16:01:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.71.05              Driver Version: 595.71.05      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   31C    P0             61W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
import os
#sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

In [4]:
import torch
import yaml
from torch.utils.data import DataLoader

In [5]:
# Training pieces: model builder, dataset, block-masking collator, trainer.
from vl_jepa.vl_jepa import build_vl_jepa
from vl_jepa.dataset import make_dummy_pairs, VLJEPADataset
from vl_jepa.masking import BlockMaskCollator
from vl_jepa.trainer import VLJEPATrainer
from vl_jepa.utils import set_seed, print_model_summary

In [6]:
# Reproducible runs.
set_seed(42)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
# The commented block shows how to shrink the model for a fast smoke test.
# Here we only shorten the training schedule (3 epochs, small batch).
# ── Config ─────────────────────────────────────────────────────────────────
with open("vl_jepa/config.yaml") as f:
    cfg = yaml.safe_load(f)

# Shrink model for quick demo - skip this part
"""
cfg["model"].update({
    "vision_embed_dim": 192,
    "vision_depth": 3,
    "vision_num_heads": 3,
    "lang_embed_dim": 128,
    "lang_depth": 2,
    "lang_num_heads": 2,
    "pred_depth": 2,
    "pred_num_heads": 3,
    "vision_proj_dim": 64,
    "lang_proj_dim": 64,
    "latent_dim": 64,
})
"""
cfg["training"].update({"epochs": 3, "batch_size": 8, "num_workers": 0, "use_amp": False})


In [9]:
# ── Model ───────────────────────────────────────────────────────────────────
model = build_vl_jepa(cfg)
print_model_summary(model)


  VL-JEPA Model Summary
  Total params   :  310,194,176
  Trainable      :  223,608,064
  Frozen (EMA)   :   86,586,112



In [10]:
# make_dummy_pairs -> synthetic pairs; BlockMaskCollator masks contiguous
# patch blocks each batch — the core self-supervised signal for VL-JEPA.
# ── Data ────────────────────────────────────────────────────────────────────
pairs = make_dummy_pairs(n=64, img_size=cfg["data"]["image_size"])
dataset = VLJEPADataset(pairs=pairs)

collator = BlockMaskCollator(
    img_size=cfg["data"]["image_size"],
    patch_size=cfg["model"]["patch_size"],
)
loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collator)

In [11]:
# Where checkpoints are written; save every SAVE_EVERY_CKPT epochs.
OUTPUT_DIR = "vljepa_model"
EPOCH = 3
LR = 1e-4
SAVE_EVERY_CKPT = 3

In [12]:
# warmup_epochs ramps the LR; use_amp toggles mixed precision.
# ── Trainer ─────────────────────────────────────────────────────────────────
trainer = VLJEPATrainer(
    model=model,
    train_loader=loader,
    lr=LR,
    epochs=EPOCH,
    warmup_epochs=1,
    use_amp=False,
    checkpoint_dir=OUTPUT_DIR,
    save_every=SAVE_EVERY_CKPT,
    log_every=1,
    device=device,
)

/home/sagemaker-user/rlvr_lwm/vl_jepa/trainer.py:91: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=use_amp)


In [13]:
# Kick off training; checkpoints land in OUTPUT_DIR.
print("\n── Starting training - ", EPOCH, "training ──")
trainer.train()
print("\nDone! Checkpoint saved to", OUTPUT_DIR)

/home/sagemaker-user/rlvr_lwm/vl_jepa/trainer.py:137: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.use_amp):



── Starting training -  3 training ──

[Epoch 1/3]  lr=1.00e-04
  [epoch   1 | step    0/8] loss=0.0041  lr=1.00e-04  mom=0.9960  t=0.8s
  [epoch   1 | step    1/8] loss=0.0011  lr=1.00e-04  mom=0.9960  t=1.0s
  [epoch   1 | step    2/8] loss=0.0004  lr=1.00e-04  mom=0.9961  t=1.2s
  [epoch   1 | step    3/8] loss=0.0002  lr=1.00e-04  mom=0.9962  t=1.3s
  [epoch   1 | step    4/8] loss=0.0001  lr=1.00e-04  mom=0.9963  t=1.5s
  [epoch   1 | step    5/8] loss=0.0001  lr=1.00e-04  mom=0.9964  t=1.7s
  [epoch   1 | step    6/8] loss=0.0001  lr=1.00e-04  mom=0.9966  t=1.9s
  [epoch   1 | step    7/8] loss=0.0001  lr=1.00e-04  mom=0.9968  t=2.1s
  train_loss=0.0008

[Epoch 2/3]  lr=1.00e-04
  [epoch   2 | step    0/8] loss=0.0001  lr=1.00e-04  mom=0.9970  t=0.2s
  [epoch   2 | step    1/8] loss=0.0001  lr=1.00e-04  mom=0.9972  t=0.4s
  [epoch   2 | step    2/8] loss=0.0001  lr=1.00e-04  mom=0.9975  t=0.5s
  [epoch   2 | step    3/8] loss=0.0001  lr=1.00e-04  mom=0.9977  t=0.7s
  [epoch   2 